In [14]:
import os
import glob
import pandas as pd
import yaml

from datasets.split_data import *

pd.set_option('display.max_rows', None)

In [17]:
# Load the configuration
with open("config.yaml", "r") as file:
    config = yaml.safe_load(file)

insect_classes = config["data_params"]["data_classes"] 
split = config["data_params"]["nature_split"] # good, meas noise, label noise percentages
src_dir = config["data_params"]["original_data_dir"]
root_dir = os.path.dirname(os.path.relpath(src_dir))

('data', 'data/phoneboxdata')

In [10]:
dest_dir = os.path.join(root_dir, f'split_{split[0]}_{split[1]}_{split[2]}')
config["data_params"]["splitted_data_dir"] = dest_dir

# Write back full config
with open("config.yaml", "w") as f:
    yaml.dump(config, f)

In [ ]:
wmv_src_folder = os.path.join(src_dir, 'wmv')
wrl_src_folder = os.path.join(src_dir, 'wrl')
trash_src_folder = os.path.join(src_dir, 'not_insect')

wmv_files = glob.glob(os.path.join(wmv_src_folder, '*.png'))
wrl_files = glob.glob(os.path.join(wrl_src_folder, '*.png'))

# Get total number of files in each class for splitting
total_samples = min(int(len(wmv_files)/100)*100, int(len(wrl_files)/100)*100)
print (f'Total samples per class: {total_samples}')

# Calculate the number of samples for each split
train_composition = {'good': split[0]/100, 'mislabel': split[1]/100, 'other': split[2]/100}

split_result = calculate_custom_splits(total_samples, train_composition)
split_result['train']['other'] = split_result['train']['mislabel']
split_result['validation']['other'] = split_result['validation']['mislabel']
print(f'Split result: {split_result}')

In [ ]:
df_wmv = get_df(wmv_src_folder)
df_wrl = get_df(wrl_src_folder)
df_trash = get_df(trash_src_folder)

df_wmv_subsets, assignment_summary_wmv = get_df_subsets(df_wmv, [
    split_result['train']['good'], 
    split_result['train']['mislabel'], 
    split_result['test']['good'], 
    split_result['validation']['good'], 
    split_result['validation']['mislabel']
    ])
df_wrl_subsets, assignment_summary_wrl = get_df_subsets(df_wrl, [
    split_result['train']['good'], 
    split_result['train']['mislabel'], 
    split_result['test']['good'], 
    split_result['validation']['good'], 
    split_result['validation']['mislabel']
    ])
df_trash_subsets, assignment_summary_trash = get_df_subsets(df_trash, [
    split_result['train']['other'], 
    split_result['train']['other'], 
    split_result['validation']['other'], 
    split_result['validation']['other'], 
    ])

print(f'Assignment summary for wmv: {assignment_summary_wmv}')
print(f'Assignment summary for wrl: {assignment_summary_wrl}')
print(f'Assignment summary for trash: {assignment_summary_trash}')

print(df_wmv_subsets['subset'].value_counts())
print(df_wrl_subsets['subset'].value_counts())
print(df_trash_subsets['subset'].value_counts())

In [ ]:
wmv_dest_folders = ['train/wmv/wmv_good', 'train/wrl/wmv_for_wrl', 'test/wmv', 'val/wmv/wmv_good', 'val/wrl/wmv_for_wrl']
wrl_dest_folders = ['train/wrl/wrl_good', 'train/wmv/wrl_for_wmv', 'test/wrl', 'val/wrl/wrl_good', 'val/wmv/wrl_for_wmv']
trash_dest_folders = ['train/wmv/wmv_trash', 'train/wrl/wrl_trash', 'val/wmv/wmv_trash', 'val/wrl/wrl_trash']

copy_files_to_dest(df_wmv_subsets, dest_dir, wmv_dest_folders)
copy_files_to_dest(df_wrl_subsets, dest_dir, wrl_dest_folders)
copy_files_to_dest(df_trash_subsets, dest_dir, trash_dest_folders)